# Fase: 04 Modeling

*Creación de modelos predictivos*

Este notebook reúne las celdas iniciales y la estructura para comenzar la creación de modelos predictivos.
Se incluyen: importaciones, localización y carga del dataset (carpeta `procesado`), vistas iniciales de EDA, y placeholders para preprocesamiento y modelado.

### Importaciones

Importaremos las librerías más comunes que usaremos durante el modelado. Si falta alguna dependencia en tu entorno, instálala (por ejemplo `xgboost`).

In [93]:
# Librerías básicas y utilidades
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

# Scikit-learn: split, preprocesado, modelos básicos y métricas
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
import joblib

# Opcional: XGBoost (no obligatorio)
try:
    import xgboost as xgb
except Exception:
    xgb = None

# Mostrar versiones (útil para reproducibilidad)
print('pandas', pd.__version__, 'numpy', np.__version__)


pandas 2.3.3 numpy 2.3.3


## Localizar y cargar el dataset desde `procesado`

El dataset procesado está en la carpeta `procesado` dentro de este directorio de `04_Modeling`. Esta celda intentará localizarla y cargar el primer CSV encontrado. Si hay varios CSV, ajusta `dataset_path` a tu archivo objetivo.

In [94]:
# Intentar localizar la carpeta 'procesado' y listar CSVs
data_dir = Path('procesado')
# Si no existe en el cwd, buscar recursivamente hacia arriba
if not data_dir.exists():
    cwd = Path.cwd()
    candidates = [p for p in cwd.rglob('procesado') if p.is_dir()]
    if candidates:
        data_dir = candidates[0]

print('Usando data_dir:', data_dir.resolve() if data_dir.exists() else data_dir)
csv_files = []
if data_dir.exists():
    csv_files = sorted(data_dir.glob('csv_procesado.csv'))

print('Archivos CSV encontrados:', csv_files)

if not csv_files:
    print('No se encontraron CSV en la ruta esperada. Verifica la carpeta `procesado`.')
else:
    dataset_path = csv_files[0]
    print('Cargando dataset:', dataset_path.name)
    df = pd.read_csv(dataset_path, low_memory=False)
    # Intentar localizar la carpeta 'procesado' y listar CSVs
data_dir = Path('procesado')
# Si no existe en el cwd, buscar recursivamente hacia arriba
if not data_dir.exists():
    cwd = Path.cwd()
    candidates = [p for p in cwd.rglob('procesado') if p.is_dir()]
    if candidates:
        data_dir = candidates[0]

print('Usando data_dir:', data_dir.resolve() if data_dir.exists() else data_dir)
csv_files = []
if data_dir.exists():
    csv_files = sorted(data_dir.glob('*.csv'))

print('Archivos CSV encontrados:', csv_files)

if not csv_files:
    print('No se encontraron CSV en la ruta esperada. Verifica la carpeta `procesado`.')
else:
    dataset_path = csv_files[0]
    print('Cargando dataset:', dataset_path.name)
    df_procesado = pd.read_csv(dataset_path, low_memory=False)
    print('Dimensiones del dataset:', df_procesado.shape)

Usando data_dir: C:\Users\diego\Desktop\Capstone_DuocUC_003V_Grupo04\Fase 3\Evidencias Extra\crisp-dm\04_Modeling\procesado
Archivos CSV encontrados: [WindowsPath('procesado/csv_procesado.csv')]
Cargando dataset: csv_procesado.csv
Usando data_dir: C:\Users\diego\Desktop\Capstone_DuocUC_003V_Grupo04\Fase 3\Evidencias Extra\crisp-dm\04_Modeling\procesado
Archivos CSV encontrados: [WindowsPath('procesado/csv_procesado.csv')]
Cargando dataset: csv_procesado.csv
Dimensiones del dataset: (132355, 37)


## Vistas rápidas
Celdas para inspeccionar el dataset: `head`, `info`, `describe`, comprobación de valores nulos y distribución de la variable objetivo.

In [95]:
# Vistas iniciales (ejecutar solo si se cargó 'df')
try:
    display(df_procesado.head())
    print('\nInfo:')
    print(df_procesado.info())
    print('\nDescripción numérica:')
    display(df_procesado.describe(include='all').T)
    print('\nConteo de valores nulos por columna:')
    display(df_procesado.isna().sum().sort_values(ascending=False).head(30))
except NameError:
    print('No se ha cargado el DataFrame `df_procesado`. Ejecuta la celda de carga del dataset primero.')

,fecha_venta,ven_id,anio,mes,dia_semana,prod_id,prod_nom,categoria,precio_venta,ven_cantidad,...,semana_ano,mes_ano,ventas_acum_prod,ingresos_acum_prod,precio_promedio_prod,rolling_7d_cantidad,rolling_30d_cantidad,ratio_valor_stock,ratio_utilidad,top10_por_ingreso
0,2024-01-10,139609,2024,1,Wednesday,11,1. CEMENTO 25K,MATERIALES CONSTRUCCION,5400.0,2.0,...,2,2024-01,2.0,11088.30,5400.0,2.0,2.0,0.0,0.0,False
1,2024-01-10,127289,2024,1,Wednesday,11,1. CEMENTO 25K,MATERIALES CONSTRUCCION,5400.0,2.0,...,2,2024-01,4.0,22967.83,5400.0,4.0,4.0,0.0,0.0,False
2,2024-01-10,175712,2024,1,Wednesday,11,1. CEMENTO 25K,MATERIALES CONSTRUCCION,5400.0,2.0,...,2,2024-01,6.0,33706.24,5400.0,6.0,6.0,0.0,0.0,False
3,2024-01-10,163392,2024,1,Wednesday,11,1. CEMENTO 25K,MATERIALES CONSTRUCCION,5400.0,2.0,...,2,2024-01,8.0,44961.56,5400.0,8.0,8.0,0.0,0.0,False
4,2024-01-10,67403,2024,1,Wednesday,11,1. CEMENTO 25K,MATERIALES CONSTRUCCION,5400.0,2.0,...,2,2024-01,10.0,56592.06,5400.0,10.0,10.0,0.0,0.0,False



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132355 entries, 0 to 132354
Data columns (total 37 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   fecha_venta            132355 non-null  object 
 1   ven_id                 132355 non-null  int64  
 2   anio                   132355 non-null  int64  
 3   mes                    132355 non-null  int64  
 4   dia_semana             132355 non-null  object 
 5   prod_id                132355 non-null  int64  
 6   prod_nom               132355 non-null  object 
 7   categoria              132355 non-null  object 
 8   precio_venta           132355 non-null  float64
 9   ven_cantidad           132355 non-null  float64
 10  ven_descuento          132355 non-null  float64
 11  ven_precio_unitario    132355 non-null  float64
 12  ingreso_neto           132355 non-null  float64
 13  suc_id                 132355 non-null  int64  
 14  sucursal               132355

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
fecha_venta,132355,138,2024-01-10,105942,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ven_id,132355.0,NaN,NaN,NaN,90761.7741,52119.667594,3.0,45603.5,91042.0,135967.5,180514.0
anio,132355.0,NaN,NaN,NaN,2024.172649,0.377945,2024.0,2024.0,2024.0,2024.0,2025.0
mes,132355.0,NaN,NaN,NaN,2.486804,3.236115,1.0,1.0,1.0,1.0,12.0
dia_semana,132355,7,Wednesday,107817,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prod_id,132355.0,NaN,NaN,NaN,4265.631076,2510.994239,11.0,2301.0,4100.0,6717.0,8066.0
prod_nom,132355,2702,GAS BUTANO 190GR,1951,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria,132355,39,GASFITERIA,29630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
precio_venta,132355.0,NaN,NaN,NaN,2655.200106,2425.913766,0.0,850.0,2000.0,3600.0,13000.0
ven_cantidad,132355.0,NaN,NaN,NaN,2.769816,2.627189,0.0,1.0,2.0,3.0,13.0



Conteo de valores nulos por columna:


fecha_venta              0
ven_id                   0
anio                     0
mes                      0
dia_semana               0
prod_id                  0
prod_nom                 0
categoria                0
precio_venta             0
ven_cantidad             0
ven_descuento            0
ven_precio_unitario      0
ingreso_neto             0
suc_id                   0
sucursal                 0
usu_id                   0
vendedor                 0
cli_id                   0
cliente                  0
stock_actual             0
valor_stock_pendiente    0
utilidad_acumulada       0
ingreso_calculado        0
pct_descuento            0
tiene_descuento          0
precio_unitario_calc     0
es_fin_de_semana         0
semana_ano               0
mes_ano                  0
ventas_acum_prod         0
dtype: int64

## Plan de preprocesamiento

- Separación automática de columnas numéricas y categóricas mediante `select_dtypes`.
- Creación de un pipeline para variables numéricas con:
  - Imputación usando `SimpleImputer(strategy='median')`
  - Escalado usando `StandardScaler()`
- Creación de un pipeline para variables categóricas con:
  - Imputación usando `SimpleImputer(strategy='most_frequent')`
  - Codificación con `OneHotEncoder`, compatible con versiones < 1.2 y ≥ 1.2 de scikit-learn
- Construcción de un `ColumnTransformer` que aplica el pipeline numérico y categórico a sus respectivas columnas.
- Construcción de un `Pipeline` general que incluye:
  - El preprocesamiento (`preprocessor`)
  - Un modelo baseline (`RandomForestClassifier`)
- Impresión de confirmación de la creación de la plantilla de preprocesamiento.
- Impresión de las listas finales `numeric_features` y `categorical_features`.
- Creación de dos copias del dataset:
  - `data_regresion` para modelos de regresión
  - `data_clasification` para modelos de clasificación

In [96]:
## =========================================
## Plan de preprocesamiento
## =========================================

# 1️⃣ Separación automática de columnas numéricas y categóricas
numeric_features = df_procesado.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = df_procesado.select_dtypes(include=['object','category','bool']).columns.tolist()

print(f"Columnas numéricas: {numeric_features}")
print(f"Columnas categóricas: {categorical_features}")

# 2️⃣ Pipeline para variables numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3️⃣ Pipeline para variables categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4️⃣ ColumnTransformer para aplicar pipelines a las columnas correctas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# 5️⃣ Pipeline general con modelo baseline (RandomForestClassifier)
pipeline_clasification = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 6️⃣ Confirmación
print("✅ Plantilla de preprocesamiento y pipeline de clasificación creada con éxito.")

# 7️⃣ Creación de copias del dataset para distintos tipos de modelos
data_regresion = df_procesado.copy()
data_clasification = df_procesado.copy()

print("✅ Copias para regresión y clasificación creadas.")

Columnas numéricas: ['ven_id', 'anio', 'mes', 'prod_id', 'precio_venta', 'ven_cantidad', 'ven_descuento', 'ven_precio_unitario', 'ingreso_neto', 'suc_id', 'usu_id', 'cli_id', 'stock_actual', 'valor_stock_pendiente', 'utilidad_acumulada', 'ingreso_calculado', 'pct_descuento', 'precio_unitario_calc', 'semana_ano', 'ventas_acum_prod', 'ingresos_acum_prod', 'precio_promedio_prod', 'rolling_7d_cantidad', 'rolling_30d_cantidad', 'ratio_valor_stock', 'ratio_utilidad']
Columnas categóricas: ['fecha_venta', 'dia_semana', 'prod_nom', 'categoria', 'sucursal', 'vendedor', 'cliente', 'tiene_descuento', 'es_fin_de_semana', 'mes_ano', 'top10_por_ingreso']
✅ Plantilla de preprocesamiento y pipeline de clasificación creada con éxito.
✅ Copias para regresión y clasificación creadas.


## Construcción de modelos predictivos

*En esta etapa se seleccionarán los algoritmos de modelado más adecuados y se entrenarán los modelos utilizando el conjunto de datos preparado. Se ajustarán los parámetros de los modelos y se evaluarán los resultados preliminares para determinar qué técnicas ofrecen el mejor desempeño para cumplir con los objetivos planteados.*

### Construcción de Modelos de Regresión Supervisados

#### Generación de variables mensuales, semanales y diarias

In [97]:
# ===================================================
# Preparación de datasets por horizonte de predicción
# ===================================================

# Partimos de data_regresion
# Mensual
data_monthly = data_regresion.groupby(['prod_id','mes_ano'], as_index=False).agg({
    'ven_cantidad':'sum',
    'precio_venta':'mean',
    'ingreso_neto':'sum'
})
# Acumulados
data_monthly['ventas_acum_prod'] = data_monthly.groupby('prod_id')['ven_cantidad'].cumsum()
# Columnas rolling necesarias (no aplicables mensual, inicializamos en 0)
data_monthly['rolling_7d_cantidad'] = 0
data_monthly['rolling_30d_cantidad'] = 0

# Semanal
data_weekly = data_regresion.groupby(['prod_id','semana_ano'], as_index=False).agg({
    'ven_cantidad':'sum',
    'precio_venta':'mean',
    'ingreso_neto':'sum'
})
data_weekly['ventas_acum_prod'] = data_weekly.groupby('prod_id')['ven_cantidad'].cumsum()
# Columnas rolling necesarias (no aplicables semanal, inicializamos en 0)
data_weekly['rolling_7d_cantidad'] = 0
data_weekly['rolling_30d_cantidad'] = 0

# Diario
data_daily = data_regresion.groupby(['prod_id','fecha_venta'], as_index=False).agg({
    'ven_cantidad':'sum',
    'precio_venta':'mean',
    'ingreso_neto':'sum'
})
data_daily = data_daily.sort_values(['prod_id','fecha_venta'])
# Rolling diarios reales
data_daily['rolling_7d_cantidad'] = data_daily.groupby('prod_id')['ven_cantidad'].rolling(7, min_periods=1).sum().reset_index(0, drop=True)
data_daily['rolling_30d_cantidad'] = data_daily.groupby('prod_id')['ven_cantidad'].rolling(30, min_periods=1).sum().reset_index(0, drop=True)
# Acumulado diario
data_daily['ventas_acum_prod'] = data_daily.groupby('prod_id')['ven_cantidad'].cumsum()

print("✅ Datasets por horizonte de predicción preparados con todas las columnas necesarias (desde data_regresion).")


✅ Datasets por horizonte de predicción preparados con todas las columnas necesarias (desde data_regresion).


#### Definición de los modelos

In [98]:
def entrenar_modelos_regresion(data, target='ven_cantidad'):
    """
    Entrena modelos de regresión (versión baseline previamente usada).

    Args:
        data: DataFrame (por ejemplo, data_monthly, data_weekly, data_daily)
        target: columna objetivo (default 'ven_cantidad')

    Returns:
        resultados: diccionario con métricas y modelos entrenados
    """
    # Separar X e y
    X = data.drop(columns=[target])
    y = data[target]

    # Columnas a usar
    num_features = ['precio_venta','ingreso_neto','ventas_acum_prod','rolling_7d_cantidad','rolling_30d_cantidad']

    # Pipelines
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # OneHotEncoder compatible
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', ohe)
    ])

    # ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_features),
        ])

    # Modelos
    modelos = {
        'RandomForest': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42))
        ]),
        'DecisionTree': DecisionTreeRegressor(random_state=42),
        'LinearRegression': LinearRegression()
    }

    resultados = {}

    for name, model in modelos.items():
        if name == 'RandomForest':
            # Split manual porque el pipeline incluye preprocesamiento
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        else:
            # Otros modelos usan solo columnas numéricas definidas
            X_num = X[num_features]
            X_train, X_test, y_train, y_test = train_test_split(X_num, y, test_size=0.2, random_state=42)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

        resultados[name] = {
            'model': model,
            'MAE': mean_absolute_error(y_test, y_pred),
            'MSE': mean_squared_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        }

    return resultados


#### Entrenamiento de modelos

In [108]:
# -------------------------------
# Entrenamiento de modelos
# -------------------------------
# Mensual
resultados_mensual = entrenar_modelos_regresion(data_monthly)
# Semanal
resultados_semanal = entrenar_modelos_regresion(data_weekly)
# Diario
resultados_diario = entrenar_modelos_regresion(data_daily)

print("Todos los modelos han sido entrenados correctamente.")

Todos los modelos han sido entrenados correctamente.


##### Generación de resultados

###### Modelos Mensuales

In [109]:
# Función para mostrar resultados sin imprimir el objeto del modelo
def mostrar_resultados(resultados, nombre_horizonte):
    print(f"\n📅 Resultados {nombre_horizonte}:")
    for model_name, metrics in resultados.items():
        print(f"\nModelo: {model_name}")
        print(f"MAE: {metrics['MAE']:.3f}")
        print(f"MSE: {metrics['MSE']:.3f}")
        print(f"R2: {metrics['R2']:.3f}")

# Mostrar los 3 horizontes
mostrar_resultados(resultados_mensual, "Mensual")



📅 Resultados Mensual:

Modelo: RandomForest
MAE: 6.105
MSE: 36960.125
R2: 0.523

Modelo: DecisionTree
MAE: 7.604
MSE: 46914.961
R2: 0.395

Modelo: LinearRegression
MAE: 31.850
MSE: 109990.491
R2: -0.419


###### Modelos Semanales

In [110]:
# Función para mostrar resultados sin imprimir el objeto del modelo
def mostrar_resultados(resultados, nombre_horizonte):
    print(f"\n📅 Resultados {nombre_horizonte}:")
    for model_name, metrics in resultados.items():
        print(f"\nModelo: {model_name}")
        print(f"MAE: {metrics['MAE']:.3f}")
        print(f"MSE: {metrics['MSE']:.3f}")
        print(f"R2: {metrics['R2']:.3f}")

# Mostrar los 3 horizontes
mostrar_resultados(resultados_semanal, "Semanal")


📅 Resultados Semanal:

Modelo: RandomForest
MAE: 4.333
MSE: 12212.927
R2: 0.504

Modelo: DecisionTree
MAE: 5.154
MSE: 13759.898
R2: 0.441

Modelo: LinearRegression
MAE: 29.619
MSE: 11968.027
R2: 0.514


###### Modelos Diarios

In [111]:
# Función para mostrar resultados sin imprimir el objeto del modelo
def mostrar_resultados(resultados, nombre_horizonte):
    print(f"\n📅 Resultados {nombre_horizonte}:")
    for model_name, metrics in resultados.items():
        print(f"\nModelo: {model_name}")
        print(f"MAE: {metrics['MAE']:.3f}")
        print(f"MSE: {metrics['MSE']:.3f}")
        print(f"R2: {metrics['R2']:.3f}")

# Mostrar los 3 horizontes
mostrar_resultados(resultados_diario, "Diario")


📅 Resultados Diario:

Modelo: RandomForest
MAE: 1.028
MSE: 90.497
R2: 0.994

Modelo: DecisionTree
MAE: 0.895
MSE: 49.672
R2: 0.996

Modelo: LinearRegression
MAE: 22.882
MSE: 7395.584
R2: 0.471


##### Guardado de modelos predictivos como pkl

In [113]:
# -------------------------------
# Guardar modelos en archivos .pkl
# -------------------------------

# Carpeta donde guardaremos los modelos
ruta_modelos = "modelos_predictivos"
os.makedirs(ruta_modelos, exist_ok=True)

def guardar_modelos(resultados, nombre_prefijo):
    """
    Guarda un diccionario de modelos en archivos .pkl
    """
    for nombre_modelo, modelo in resultados.items():
        ruta_archivo = os.path.join(ruta_modelos, f"{nombre_prefijo}_{nombre_modelo}.pkl")
        joblib.dump(modelo, ruta_archivo)
        print(f"Guardado: {ruta_archivo}")

# Guardar modelos por frecuencia
guardar_modelos(resultados_mensual, "mensual")
guardar_modelos(resultados_semanal, "semanal")
guardar_modelos(resultados_diario, "diario")

print("Todos los modelos han sido guardados correctamente.")

Guardado: modelos_predictivos\mensual_RandomForest.pkl
Guardado: modelos_predictivos\mensual_DecisionTree.pkl
Guardado: modelos_predictivos\mensual_LinearRegression.pkl
Guardado: modelos_predictivos\semanal_RandomForest.pkl
Guardado: modelos_predictivos\semanal_DecisionTree.pkl
Guardado: modelos_predictivos\semanal_LinearRegression.pkl
Guardado: modelos_predictivos\diario_RandomForest.pkl
Guardado: modelos_predictivos\diario_DecisionTree.pkl
Guardado: modelos_predictivos\diario_LinearRegression.pkl
Todos los modelos han sido guardados correctamente.
